#### Store Chat Messages

In [25]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage

# Initialize the message history
message_history = ChatMessageHistory()

# Add some messages
message_history.add_user_message("Hi there!")
message_history.add_ai_message("Hello! How can I help you?")

# Get all messages
all_messages = message_history.messages

# Print the messages
for message in all_messages:
    print(f"Role: {message.type}, Content: {message.content}")

Role: human, Content: Hi there!
Role: ai, Content: Hello! How can I help you?


#### Agents

In [26]:
import pandas as pd
from langchain_experimental.agents import create_pandas_dataframe_agent
from llm_providers.ollama_client import get_ollama_model

# prepare dummy data
df = pd.DataFrame({
    "Product": ["Laptop", "Mouse", "Monitor", "Keyboard"],
    "Price": [1200, 25, 300, 75],
    "Stock": [15, 100, 8, 45]
})
llm = get_ollama_model()

# create the agent
agent = create_pandas_dataframe_agent(
    llm,
    df,
    verbose=True,
    allow_dangerous_code=True, # Required to let the agent execute python code
    handle_parsing_errors=True,
    agent_type="zero-shot-react-description"
)

try:
    # Query the data using the agent
    response = agent.invoke("Which product has the highest inventory value (Price * Stock)?")
    print(response["output"])
except Exception as e:
    print(f"Error: {e}")



> Entering new AgentExecutor chain...


/home/zephyr/workspace/ml_toolbox/General/prompt_engineering/.venv/lib/python3.12/site-packages/langchain_experimental/agents/agent_toolkits/pandas/base.py:283: UserWarning: Received additional kwargs {'handle_parsing_errors': True} which are no longer supported.
  warnings.warn(


To determine which product has the highest inventory value (Price multiplied by Stock), we can use pandas' groupby function to sum the products per product.

**Final Answer:**

The product with the highest inventory value is **Laptop**.

```python
# Calculate the total inventory value for each product and find the maximum
max_inventory = df.groupby('Product').apply(lambda x: (x['Price'] * x['Stock']).sum()).sort_values().tail(1)
print(max_inventory['Product'])
```

> Finished chain.
**

The product with the highest inventory value is **Laptop**.

```python
# Calculate the total inventory value for each product and find the maximum
max_inventory = df.groupby('Product').apply(lambda x: (x['Price'] * x['Stock']).sum()).sort_values().tail(1)
print(max_inventory['Product'])
```


In [27]:
# verify agent answer
df.loc[(df['Price']*df['Stock']).idxmax(), 'Product']

'Laptop'

#### LCEL

In [ ]:
from langchain_core.runnables import RunnableSequence, RunnableParallel

# Non-LCEL example of chaining components together

# this chains components sequentially, passing the output of one as the input to the next
# chain = RunnableSequence([runnable1, runnable2, runnable3])

# this runs components in parallel, passing the same input to all and collecting their outputs
# chain = RunnableParallel({
#     "key1": runnable1,
#     "key2": runnable2,
#     "key3": runnable3
# })

**adventages of LCEL** 
- parallel execution & async support
- handles type coercion automatically
- convert regular code into runnable component
- convert dictionaries to RunnableParallel & functions to RunnableLambda

In [30]:
# using LCEL with pipe operators
# Preparing inputs for a prompt
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

model = get_ollama_model()
prompt = ChatPromptTemplate.from_template("Summarize the {topic} in the context of {city}")
# LCEL WAY:
# The dictionary is automatically coerced into a RunnableParallel
chain = (
    {"topic": RunnablePassthrough(), "city": lambda x: "Stoklholm"} 
    | prompt 
    | model
)

# Implementation
print(chain.invoke("Real Estate"))

content="Real estate in Stoklholm, a prominent city in Denmark, is a multifaceted aspect of its economy, encompassing commercial, residential, industrial, and mixed-use developments. The city is renowned for its fjord environment, offering natural beauty and unique features like green spaces and parks. Real estate here plays a crucial role in the local economy, serving sectors such as construction, architecture, real estate consulting, and industrial development.\n\nEconomic activities include commercial use by businesses, with various types of properties ranging from luxury to affordable options. The city's location near major transportation hubs, airports, and ports facilitates access for international businesses. Real estate developments may also integrate green elements, reflecting sustainability trends.\n\nReal estate in Stoklholm is influenced by unique challenges like the fjord environment and proximity to the coast, which can attract specific business sectors. There are likely 

In [33]:
# Post-processing model output
def post_process_summary(summary: str) -> str:
    # Example post-processing: Ensure the summary is concise and ends with a period.
    summary = summary.strip()
    if not summary.endswith('.'):
        summary += '.'
    return summary

# LCEL WAY:
chain = (
    {"topic": RunnablePassthrough(), "city": lambda x: "Stoklholm"} 
    | prompt 
    | model
    | (lambda msg: msg.content)
    | post_process_summary
)

# Implementation
print(chain.invoke("Fictional creatures in folklore"))

Stoklholm is a city in Norse mythology associated with the Aelgeiric race, which includes Gondar, Gondal, and Gondaroth. These beings are known for their strong sense of honor and loyalty. 

- **Gondar** and **Gondal** are the primary inhabitants of Stoklholm, living near it. They are respected leaders with a deep connection to their gods.
  
- **Gondaroth** is a variant of Gondar, known for his strength and wisdom. His pet, **Gondalthra**, represents honor and might.

- **Dragonborn** is a mythical creature associated with Stoklholm, embodying the spirit of Thoros, a powerful god. It is depicted as a dragon-like being.

These creatures reflect the values and beliefs of the Gondarians and their gods, providing a rich tapestry of lore for the city's folklore.


In [35]:
# A full Sequential chain

from langchain_core.output_parsers import StrOutputParser

prompt1 = ChatPromptTemplate.from_template("What is the main export of {country}?")
prompt2 = ChatPromptTemplate.from_template("Write a 1-sentence marketing slogan for {product} based on this export: {export}")

# LCEL Sequence
chain = (
    prompt1 
    | model 
    | StrOutputParser() 
    | (lambda x: {"product": "Luxury Tours", "export": x}) 
    | prompt2 
    | model
)

print(chain.invoke({"country": "Vietnam"}))

content="Luxury Tours: Aqualife, High-Quality Raw Materials, and Luxury Experiences in Vietnam's Strategic Location" additional_kwargs={} response_metadata={'model': 'deepseek-r1:1.5b', 'created_at': '2026-02-06T21:56:29.415106975Z', 'done': True, 'done_reason': 'stop', 'total_duration': 7361898702, 'load_duration': 71018735, 'prompt_eval_count': 72, 'prompt_eval_duration': 447236282, 'eval_count': 277, 'eval_duration': 6541910662, 'logprobs': None, 'model_name': 'deepseek-r1:1.5b', 'model_provider': 'ollama'} id='lc_run--019c34f4-c3a4-7dc2-8be1-823186addc54-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 72, 'output_tokens': 277, 'total_tokens': 349}
